In [1]:
# ---------------- 6.5) Linear-Closed baseline (closed-form bridge; ATE, plug-in SE) ----------------
# 公式： J_hat = E_n[Tpsi]^T · (E_n[phi psi^T])^+ · E_n[Y phi]
# plug-in 方差：以 REG 形式的逐样本 g_i = (Tpsi_i)^T theta_h 构造 sd(g_i - mean)/sqrt(n)

import numpy as np


import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F, torch.optim as optim
from math import sqrt

# ---------------- 1) Load & preprocess ----------------
url = "https://hbiostat.org/data/repo/rhc.csv"   # Vanderbilt Biostatistics RHC
df = pd.read_csv(url)

# A = RHC 指示；Y = 30-day survival (和论文表方向一致)
A = (df["swang1"].astype(str).str.upper().str.strip() == "RHC").astype(np.float32).values[:, None]
Y = (df["dth30"].astype(str).str.upper().str.strip() == "NO").astype(np.float32).values[:, None]

# 负控变量（与论文设置一致）
Z_cols = ["pafi1", "paco21"]   # NC exposures
W_cols = ["ph1",   "hema1"]    # NC outcomes





from itertools import combinations
# 所有变量
all_cols = ["pafi1", "paco21", "ph1", "hema1"]
# 生成所有 6 种分配方案：从 4 个里选 2 个做 Z，其余做 W
schemes = []
for z_pair in combinations(all_cols, 2):
    w_pair = [col for col in all_cols if col not in z_pair]
    schemes.append((list(z_pair), w_pair))
# === 这里用一个参数来控制采用哪一种方案（1~6）===
scheme_id =  1  # 改成 1,2,3,4,5,6 之一
if not (1 <= scheme_id <= len(schemes)):
    raise ValueError(f"scheme_id 必须在 1~{len(schemes)} 之间")
Z_cols, W_cols = schemes[scheme_id - 1]
print(f"当前使用的方案 {scheme_id}:")
print("Z_cols =", Z_cols)
print("W_cols =", W_cols)





for c in Z_cols + W_cols:
    if c not in df.columns:
        raise ValueError(f"Missing column {c} in RHC csv")

# X = baseline 协变量（除去 A、Y、W、Z 及明显非特征列）
drop_cols = ["ptid","sadmdte","dschdte","dthdte","lstctdte","t3d30","swang1","dth30"] + Z_cols + W_cols
X_df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors="ignore")

# One-hot（drop_first 以避免共线），均值填补 + 标准化
num_cols = X_df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X_df.columns if c not in num_cols]
X = pd.get_dummies(X_df, columns=cat_cols, drop_first=True)
X = X.replace([np.inf, -np.inf], np.nan).fillna(X.mean()).astype(np.float32)
X = ((X - X.mean())/X.std(ddof=0)).fillna(0.0)

# W, Z 简单均值填补 + 标准化
def stdize(M):
    M = M.astype(np.float32).copy()
    m = np.nanmean(M, axis=0); M[np.isnan(M)] = np.take(m, np.where(np.isnan(M))[1])
    M -= M.mean(0); M /= (M.std(0, ddof=0) + 1e-8)
    return M

Z = stdize(df[Z_cols].values)
W = stdize(df[W_cols].values)

X = X.values.astype(np.float32)
A = A.astype(np.float32); Y = Y.astype(np.float32)

n, dx = X.shape
dz, dw = Z.shape[1], W.shape[1]
print(f"Loaded RHC: n={n}, dimX={dx}, dimZ={dz}, dimW={dw}")



# 使用 numpy.float64 提高数值稳定性
X_np = X.astype(np.float64)
Z_np = Z.astype(np.float64)
W_np = W.astype(np.float64)
A_np = A.astype(np.float64)
Y_np = Y.astype(np.float64)

n = X_np.shape[0]
ones = np.ones((n, 1), dtype=np.float64)

# 线性基
b_zx = np.concatenate([ones, Z_np, X_np], axis=1)  # n x d1
b_wx = np.concatenate([ones, W_np, X_np], axis=1)  # n x d2
d1, d2 = b_zx.shape[1], b_wx.shape[1]

# 分块特征（phi, psi）
phi_lin = np.concatenate([(1.0 - A_np) * b_zx, A_np * b_zx], axis=1)   # n x (2*d1)
psi_lin = np.concatenate([(1.0 - A_np) * b_wx, A_np * b_wx], axis=1)   # n x (2*d2)

# Tpsi（与 A 无关）：二元 ATE 下相当于 “(a=1) − (a=0)”
Tpsi_lin = np.concatenate([-b_wx, +b_wx], axis=1)                      # n x (2*d2)

# 样本矩
M_hat = (phi_lin.T @ psi_lin) / n                     # (2*d1) x (2*d2)
v_hat = (phi_lin * Y_np).mean(axis=0)                 # (2*d1,)
c_hat = Tpsi_lin.mean(axis=0)                         # (2*d2,)

# 闭式求解
M_pinv   = np.linalg.pinv(M_hat, rcond=1e-8)          # (2*d2) x (2*d1)
theta_h  = M_pinv @ v_hat                             # (2*d2,) —— h 的线性系数
g_per    = Tpsi_lin @ theta_h                         # (n,)    —— 逐样本贡献 (Th)(W,X)

LC_point = float(g_per.mean())
psi_lc   = g_per - LC_point
LC_se    = float(np.std(psi_lc, ddof=1) / np.sqrt(n))
LC_ci    = (LC_point - 1.96 * LC_se, LC_point + 1.96 * LC_se)

print("---------------------------------------------------------------")
print(f"Linear-closed : {LC_point:+.4f} ({LC_se:.5f})")
print(f"95% CIs       : [{LC_ci[0]:+.4f}, {LC_ci[1]:+.4f}]")
print("===============================================================")









############################第一种setting

import numpy as np
import pandas as pd
from linearmodels.iv import IV2SLS

formula = "Y ~ 1 + A + X + [W ~ Z]"

iv_model = IV2SLS.from_formula(formula, data=df)
iv_res = iv_model.fit(cov_type="robust")
############################第一种setting
# print("original:")

# A 是 exog 中第 2 列（intercept, A, X...）
beta_A = iv_res.params[1]
se_A   = iv_res.std_errors[1]

# print("A coefficient:", beta_A)
# print("Std Error    :", se_A)
print(f"2sls(original): {beta_A:.4f} ({se_A:.5f})")
print(f"{beta_A:.4f} ({se_A:.5f}) [{beta_A - 1.96*se_A:.4f}, {beta_A + 1.96*se_A:.4f}]")


############################第二种setting


# ================== Step 1: select top-3 X by correlation with Y ==================

# Y 是 (n, 1)，拉平
Y_flat = Y.reshape(-1)

# 计算每个 X_j 与 Y 的皮尔逊相关（绝对值）
corrs = np.array([
    np.corrcoef(X[:, j], Y_flat)[0, 1] for j in range(X.shape[1])
])

# 处理极端情况（常数列等）
corrs = np.nan_to_num(corrs, nan=0.0)

# 选绝对值最大的 3 个
top_k = 2
top_idx = np.argsort(np.abs(corrs))[-top_k:][::-1]

# print("Top-3 X indices (by |corr(X,Y)|):", top_idx)
# print("Top-3 correlations:", corrs[top_idx])

# 取子矩阵
X_top = X[:, top_idx]    # shape (n, 3)


# ================== Step 2: construct interaction terms ==================

# W: (n, 2), X_top: (n, 3)
WX_top = np.hstack([
    W[:, j:j+1] * X_top for j in range(W.shape[1])
])   # shape (n, 2*3 = 6)

# Z: (n, 2)
ZX_top = np.hstack([
    Z[:, j:j+1] * X_top for j in range(Z.shape[1])
])   # shape (n, 2*3 = 6)

# ================== Step 3: IV 2SLS with reduced interactions ==================

from linearmodels.iv import IV2SLS

# exogenous regressors: [1, A, X]
exog = np.hstack([
    np.ones((len(Y), 1)),
    A.reshape(-1, 1),
    X
])

# endogenous regressors: [W, W×X_top]
endog = np.hstack([
    W,
    WX_top
])

# instruments: [Z, Z×X_top]
instruments = np.hstack([
    Z,
    ZX_top
])

iv_res = IV2SLS(
    dependent=Y,
    exog=exog,
    endog=endog,
    instruments=instruments
).fit(cov_type="robust")

# A 是 exog 的第 2 列（[1, A, X...]）
beta_A = iv_res.params[1]
se_A   = iv_res.std_errors[1]

print("==============================================")
# print("2SLS with reduced interactions (top-3 X)")
print(f"2sls(interact-top3): {beta_A:.4f} ({se_A:.5f})")
print(f"{beta_A:.4f} ({se_A:.5f}) [{beta_A - 1.96*se_A:.4f}, {beta_A + 1.96*se_A:.4f}]")

print("==============================================")












# ######################第三种setting
# import numpy as np
# from linearmodels.iv import IV2SLS
# import statsmodels.api as sm

# 手动构造平方项
X_top2 = X_top ** 2
W2 = W ** 2
Z2 = Z ** 2

exog = np.hstack([
    np.ones((len(Y), 1)),
    A.reshape(-1, 1),
    X,
   # X_top2
])

endog = np.hstack([
    W, 
    W2
    
])

instruments = np.hstack([
    Z,
    Z2
    
])

iv_res = IV2SLS(
    dependent=Y,
    exog=exog,
    endog=endog,
    instruments=instruments
).fit(cov_type="robust")

beta_A = iv_res.params[1]
se_A   = iv_res.std_errors[1]
# print(f"2sls(interact-top3): {beta_A:.4f} ({se_A:.5f})")
print(f"2sls(quar): {beta_A:.4f} ({se_A:.5f})")
print(f"{beta_A:.4f} ({se_A:.5f}) [{beta_A - 1.96*se_A:.4f}, {beta_A + 1.96*se_A:.4f}]")


############################第四种setting
# Interaction + Quadratic IV
# Y ~ A + X + [W, W^2, W*X ~ Z, Z^2, Z*X]

# 交互项
WX = np.hstack([
    W[:, j:j+1] * X for j in range(W.shape[1])
])

ZX = np.hstack([
    Z[:, j:j+1] * X for j in range(Z.shape[1])
])

# 二次项
W2 = W ** 2
Z2 = Z ** 2

# exogenous regressors
exog = np.hstack([
    np.ones((len(Y), 1)),   # intercept
    A.reshape(-1, 1),
    X
])

# endogenous regressors
endog = np.hstack([
    W,
    W2,
    WX_top
])

# instruments
instruments = np.hstack([
    Z,
    Z2,
    ZX_top
])

iv_res = IV2SLS(
    dependent=Y,
    exog=exog,
    endog=endog,
    instruments=instruments
).fit(cov_type="robust")

beta_A = iv_res.params[1]
se_A   = iv_res.std_errors[1]
print("==============================================")
print(f"2sls(interact-quar-both): {beta_A:.4f} ({se_A:.5f})")
print(f"{beta_A:.4f} ({se_A:.5f}) [{beta_A - 1.96*se_A:.4f}, {beta_A + 1.96*se_A:.4f}]")






当前使用的方案 1:
Z_cols = ['pafi1', 'paco21']
W_cols = ['ph1', 'hema1']
Loaded RHC: n=5735, dimX=70, dimZ=2, dimW=2
---------------------------------------------------------------
Linear-closed : -0.0732 (0.00454)
95% CIs       : [-0.0821, -0.0643]


/var/folders/6m/31x0jbln169_6g4fn60b2dsc0000gn/T/ipykernel_18637/4227321194.py:147: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  beta_A = iv_res.params[1]
/var/folders/6m/31x0jbln169_6g4fn60b2dsc0000gn/T/ipykernel_18637/4227321194.py:148: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  se_A   = iv_res.std_errors[1]
/var/folders/6m/31x0jbln169_6g4fn60b2dsc0000gn/T/ipykernel_18637/4227321194.py:226: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.ilo

2sls(original): -0.0666 (0.02282)
-0.0666 (0.02282) [-0.1113, -0.0219]
2sls(interact-top3): -0.0505 (0.03460)
-0.0505 (0.03460) [-0.1183, 0.0173]
2sls(quar): -0.0500 (0.02460)
-0.0500 (0.02460) [-0.0982, -0.0018]
2sls(interact-quar-both): -0.0647 (0.04941)
-0.0647 (0.04941) [-0.1616, 0.0321]


/var/folders/6m/31x0jbln169_6g4fn60b2dsc0000gn/T/ipykernel_18637/4227321194.py:283: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  beta_A = iv_res.params[1]
/var/folders/6m/31x0jbln169_6g4fn60b2dsc0000gn/T/ipykernel_18637/4227321194.py:284: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  se_A   = iv_res.std_errors[1]
/var/folders/6m/31x0jbln169_6g4fn60b2dsc0000gn/T/ipykernel_18637/4227321194.py:335: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.ilo

In [2]:
# ---------------- 6.5) Linear-Closed baseline (closed-form bridge; ATE, plug-in SE) ----------------
# 公式： J_hat = E_n[Tpsi]^T · (E_n[phi psi^T])^+ · E_n[Y phi]
# plug-in 方差：以 REG 形式的逐样本 g_i = (Tpsi_i)^T theta_h 构造 sd(g_i - mean)/sqrt(n)

import numpy as np


import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F, torch.optim as optim
from math import sqrt

# ---------------- 1) Load & preprocess ----------------
url = "https://hbiostat.org/data/repo/rhc.csv"   # Vanderbilt Biostatistics RHC
df = pd.read_csv(url)

# A = RHC 指示；Y = 30-day survival (和论文表方向一致)
A = (df["swang1"].astype(str).str.upper().str.strip() == "RHC").astype(np.float32).values[:, None]
Y = (df["dth30"].astype(str).str.upper().str.strip() == "NO").astype(np.float32).values[:, None]

# 负控变量（与论文设置一致）
Z_cols = ["pafi1", "paco21"]   # NC exposures
W_cols = ["ph1",   "hema1"]    # NC outcomes





from itertools import combinations
# 所有变量
all_cols = ["pafi1", "paco21", "ph1", "hema1"]
# 生成所有 6 种分配方案：从 4 个里选 2 个做 Z，其余做 W
schemes = []
for z_pair in combinations(all_cols, 2):
    w_pair = [col for col in all_cols if col not in z_pair]
    schemes.append((list(z_pair), w_pair))
# === 这里用一个参数来控制采用哪一种方案（1~6）===
scheme_id =  2  # 改成 1,2,3,4,5,6 之一
if not (1 <= scheme_id <= len(schemes)):
    raise ValueError(f"scheme_id 必须在 1~{len(schemes)} 之间")
Z_cols, W_cols = schemes[scheme_id - 1]
print(f"当前使用的方案 {scheme_id}:")
print("Z_cols =", Z_cols)
print("W_cols =", W_cols)





for c in Z_cols + W_cols:
    if c not in df.columns:
        raise ValueError(f"Missing column {c} in RHC csv")

# X = baseline 协变量（除去 A、Y、W、Z 及明显非特征列）
drop_cols = ["ptid","sadmdte","dschdte","dthdte","lstctdte","t3d30","swang1","dth30"] + Z_cols + W_cols
X_df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors="ignore")

# One-hot（drop_first 以避免共线），均值填补 + 标准化
num_cols = X_df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X_df.columns if c not in num_cols]
X = pd.get_dummies(X_df, columns=cat_cols, drop_first=True)
X = X.replace([np.inf, -np.inf], np.nan).fillna(X.mean()).astype(np.float32)
X = ((X - X.mean())/X.std(ddof=0)).fillna(0.0)

# W, Z 简单均值填补 + 标准化
def stdize(M):
    M = M.astype(np.float32).copy()
    m = np.nanmean(M, axis=0); M[np.isnan(M)] = np.take(m, np.where(np.isnan(M))[1])
    M -= M.mean(0); M /= (M.std(0, ddof=0) + 1e-8)
    return M

Z = stdize(df[Z_cols].values)
W = stdize(df[W_cols].values)

X = X.values.astype(np.float32)
A = A.astype(np.float32); Y = Y.astype(np.float32)

n, dx = X.shape
dz, dw = Z.shape[1], W.shape[1]
print(f"Loaded RHC: n={n}, dimX={dx}, dimZ={dz}, dimW={dw}")



# 使用 numpy.float64 提高数值稳定性
X_np = X.astype(np.float64)
Z_np = Z.astype(np.float64)
W_np = W.astype(np.float64)
A_np = A.astype(np.float64)
Y_np = Y.astype(np.float64)

n = X_np.shape[0]
ones = np.ones((n, 1), dtype=np.float64)

# 线性基
b_zx = np.concatenate([ones, Z_np, X_np], axis=1)  # n x d1
b_wx = np.concatenate([ones, W_np, X_np], axis=1)  # n x d2
d1, d2 = b_zx.shape[1], b_wx.shape[1]

# 分块特征（phi, psi）
phi_lin = np.concatenate([(1.0 - A_np) * b_zx, A_np * b_zx], axis=1)   # n x (2*d1)
psi_lin = np.concatenate([(1.0 - A_np) * b_wx, A_np * b_wx], axis=1)   # n x (2*d2)

# Tpsi（与 A 无关）：二元 ATE 下相当于 “(a=1) − (a=0)”
Tpsi_lin = np.concatenate([-b_wx, +b_wx], axis=1)                      # n x (2*d2)

# 样本矩
M_hat = (phi_lin.T @ psi_lin) / n                     # (2*d1) x (2*d2)
v_hat = (phi_lin * Y_np).mean(axis=0)                 # (2*d1,)
c_hat = Tpsi_lin.mean(axis=0)                         # (2*d2,)

# 闭式求解
M_pinv   = np.linalg.pinv(M_hat, rcond=1e-8)          # (2*d2) x (2*d1)
theta_h  = M_pinv @ v_hat                             # (2*d2,) —— h 的线性系数
g_per    = Tpsi_lin @ theta_h                         # (n,)    —— 逐样本贡献 (Th)(W,X)

LC_point = float(g_per.mean())
psi_lc   = g_per - LC_point
LC_se    = float(np.std(psi_lc, ddof=1) / np.sqrt(n))
LC_ci    = (LC_point - 1.96 * LC_se, LC_point + 1.96 * LC_se)

print("---------------------------------------------------------------")
print(f"Linear-closed : {LC_point:+.4f} ({LC_se:.5f})")
print(f"95% CIs       : [{LC_ci[0]:+.4f}, {LC_ci[1]:+.4f}]")
print("===============================================================")









############################第一种setting

import numpy as np
import pandas as pd
from linearmodels.iv import IV2SLS

formula = "Y ~ 1 + A + X + [W ~ Z]"

iv_model = IV2SLS.from_formula(formula, data=df)
iv_res = iv_model.fit(cov_type="robust")
############################第一种setting
# print("original:")

# A 是 exog 中第 2 列（intercept, A, X...）
beta_A = iv_res.params[1]
se_A   = iv_res.std_errors[1]

# print("A coefficient:", beta_A)
# print("Std Error    :", se_A)
print(f"2sls(original): {beta_A:.4f} ({se_A:.5f})")
print(f"{beta_A:.4f} ({se_A:.5f}) [{beta_A - 1.96*se_A:.4f}, {beta_A + 1.96*se_A:.4f}]")


############################第二种setting


# ================== Step 1: select top-3 X by correlation with Y ==================

# Y 是 (n, 1)，拉平
Y_flat = Y.reshape(-1)

# 计算每个 X_j 与 Y 的皮尔逊相关（绝对值）
corrs = np.array([
    np.corrcoef(X[:, j], Y_flat)[0, 1] for j in range(X.shape[1])
])

# 处理极端情况（常数列等）
corrs = np.nan_to_num(corrs, nan=0.0)

# 选绝对值最大的 3 个
top_k = 2
top_idx = np.argsort(np.abs(corrs))[-top_k:][::-1]

# print("Top-3 X indices (by |corr(X,Y)|):", top_idx)
# print("Top-3 correlations:", corrs[top_idx])

# 取子矩阵
X_top = X[:, top_idx]    # shape (n, 3)


# ================== Step 2: construct interaction terms ==================

# W: (n, 2), X_top: (n, 3)
WX_top = np.hstack([
    W[:, j:j+1] * X_top for j in range(W.shape[1])
])   # shape (n, 2*3 = 6)

# Z: (n, 2)
ZX_top = np.hstack([
    Z[:, j:j+1] * X_top for j in range(Z.shape[1])
])   # shape (n, 2*3 = 6)

# ================== Step 3: IV 2SLS with reduced interactions ==================

from linearmodels.iv import IV2SLS

# exogenous regressors: [1, A, X]
exog = np.hstack([
    np.ones((len(Y), 1)),
    A.reshape(-1, 1),
    X
])

# endogenous regressors: [W, W×X_top]
endog = np.hstack([
    W,
    WX_top
])

# instruments: [Z, Z×X_top]
instruments = np.hstack([
    Z,
    ZX_top
])

iv_res = IV2SLS(
    dependent=Y,
    exog=exog,
    endog=endog,
    instruments=instruments
).fit(cov_type="robust")

# A 是 exog 的第 2 列（[1, A, X...]）
beta_A = iv_res.params[1]
se_A   = iv_res.std_errors[1]

print("==============================================")
# print("2SLS with reduced interactions (top-3 X)")
print(f"2sls(interact-top3): {beta_A:.4f} ({se_A:.5f})")
print(f"{beta_A:.4f} ({se_A:.5f}) [{beta_A - 1.96*se_A:.4f}, {beta_A + 1.96*se_A:.4f}]")

print("==============================================")












# ######################第三种setting
# import numpy as np
# from linearmodels.iv import IV2SLS
# import statsmodels.api as sm

# 手动构造平方项
X_top2 = X_top ** 2
W2 = W ** 2
Z2 = Z ** 2

exog = np.hstack([
    np.ones((len(Y), 1)),
    A.reshape(-1, 1),
    X,
   # X_top2
])

endog = np.hstack([
    W, 
    W2
    
])

instruments = np.hstack([
    Z,
    Z2
    
])

iv_res = IV2SLS(
    dependent=Y,
    exog=exog,
    endog=endog,
    instruments=instruments
).fit(cov_type="robust")

beta_A = iv_res.params[1]
se_A   = iv_res.std_errors[1]
# print(f"2sls(interact-top3): {beta_A:.4f} ({se_A:.5f})")
print(f"2sls(quar): {beta_A:.4f} ({se_A:.5f})")
print(f"{beta_A:.4f} ({se_A:.5f}) [{beta_A - 1.96*se_A:.4f}, {beta_A + 1.96*se_A:.4f}]")


############################第四种setting
# Interaction + Quadratic IV
# Y ~ A + X + [W, W^2, W*X ~ Z, Z^2, Z*X]

# 交互项
WX = np.hstack([
    W[:, j:j+1] * X for j in range(W.shape[1])
])

ZX = np.hstack([
    Z[:, j:j+1] * X for j in range(Z.shape[1])
])

# 二次项
W2 = W ** 2
Z2 = Z ** 2

# exogenous regressors
exog = np.hstack([
    np.ones((len(Y), 1)),   # intercept
    A.reshape(-1, 1),
    X
])

# endogenous regressors
endog = np.hstack([
    W,
    W2,
    WX_top
])

# instruments
instruments = np.hstack([
    Z,
    Z2,
    ZX_top
])

iv_res = IV2SLS(
    dependent=Y,
    exog=exog,
    endog=endog,
    instruments=instruments
).fit(cov_type="robust")

beta_A = iv_res.params[1]
se_A   = iv_res.std_errors[1]
print("==============================================")
print(f"2sls(interact-quar-both): {beta_A:.4f} ({se_A:.5f})")
print(f"{beta_A:.4f} ({se_A:.5f}) [{beta_A - 1.96*se_A:.4f}, {beta_A + 1.96*se_A:.4f}]")






当前使用的方案 2:
Z_cols = ['pafi1', 'ph1']
W_cols = ['paco21', 'hema1']
Loaded RHC: n=5735, dimX=70, dimZ=2, dimW=2
---------------------------------------------------------------
Linear-closed : -0.0644 (0.01409)
95% CIs       : [-0.0920, -0.0368]


/var/folders/6m/31x0jbln169_6g4fn60b2dsc0000gn/T/ipykernel_18637/1731871455.py:147: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  beta_A = iv_res.params[1]
/var/folders/6m/31x0jbln169_6g4fn60b2dsc0000gn/T/ipykernel_18637/1731871455.py:148: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  se_A   = iv_res.std_errors[1]
/var/folders/6m/31x0jbln169_6g4fn60b2dsc0000gn/T/ipykernel_18637/1731871455.py:226: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.ilo

2sls(original): -0.0633 (0.02585)
-0.0633 (0.02585) [-0.1140, -0.0127]
2sls(interact-top3): -0.0772 (0.11825)
-0.0772 (0.11825) [-0.3090, 0.1546]
2sls(quar): -0.0336 (0.02301)
-0.0336 (0.02301) [-0.0787, 0.0115]
2sls(interact-quar-both): 0.0386 (0.21513)
0.0386 (0.21513) [-0.3831, 0.4602]


/var/folders/6m/31x0jbln169_6g4fn60b2dsc0000gn/T/ipykernel_18637/1731871455.py:283: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  beta_A = iv_res.params[1]
/var/folders/6m/31x0jbln169_6g4fn60b2dsc0000gn/T/ipykernel_18637/1731871455.py:284: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  se_A   = iv_res.std_errors[1]
/var/folders/6m/31x0jbln169_6g4fn60b2dsc0000gn/T/ipykernel_18637/1731871455.py:335: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.ilo

In [3]:
# ---------------- 6.5) Linear-Closed baseline (closed-form bridge; ATE, plug-in SE) ----------------
# 公式： J_hat = E_n[Tpsi]^T · (E_n[phi psi^T])^+ · E_n[Y phi]
# plug-in 方差：以 REG 形式的逐样本 g_i = (Tpsi_i)^T theta_h 构造 sd(g_i - mean)/sqrt(n)

import numpy as np


import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F, torch.optim as optim
from math import sqrt

# ---------------- 1) Load & preprocess ----------------
url = "https://hbiostat.org/data/repo/rhc.csv"   # Vanderbilt Biostatistics RHC
df = pd.read_csv(url)

# A = RHC 指示；Y = 30-day survival (和论文表方向一致)
A = (df["swang1"].astype(str).str.upper().str.strip() == "RHC").astype(np.float32).values[:, None]
Y = (df["dth30"].astype(str).str.upper().str.strip() == "NO").astype(np.float32).values[:, None]

# 负控变量（与论文设置一致）
Z_cols = ["pafi1", "paco21"]   # NC exposures
W_cols = ["ph1",   "hema1"]    # NC outcomes





from itertools import combinations
# 所有变量
all_cols = ["pafi1", "paco21", "ph1", "hema1"]
# 生成所有 6 种分配方案：从 4 个里选 2 个做 Z，其余做 W
schemes = []
for z_pair in combinations(all_cols, 2):
    w_pair = [col for col in all_cols if col not in z_pair]
    schemes.append((list(z_pair), w_pair))
# === 这里用一个参数来控制采用哪一种方案（1~6）===
scheme_id =  3  # 改成 1,2,3,4,5,6 之一
if not (1 <= scheme_id <= len(schemes)):
    raise ValueError(f"scheme_id 必须在 1~{len(schemes)} 之间")
Z_cols, W_cols = schemes[scheme_id - 1]
print(f"当前使用的方案 {scheme_id}:")
print("Z_cols =", Z_cols)
print("W_cols =", W_cols)





for c in Z_cols + W_cols:
    if c not in df.columns:
        raise ValueError(f"Missing column {c} in RHC csv")

# X = baseline 协变量（除去 A、Y、W、Z 及明显非特征列）
drop_cols = ["ptid","sadmdte","dschdte","dthdte","lstctdte","t3d30","swang1","dth30"] + Z_cols + W_cols
X_df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors="ignore")

# One-hot（drop_first 以避免共线），均值填补 + 标准化
num_cols = X_df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X_df.columns if c not in num_cols]
X = pd.get_dummies(X_df, columns=cat_cols, drop_first=True)
X = X.replace([np.inf, -np.inf], np.nan).fillna(X.mean()).astype(np.float32)
X = ((X - X.mean())/X.std(ddof=0)).fillna(0.0)

# W, Z 简单均值填补 + 标准化
def stdize(M):
    M = M.astype(np.float32).copy()
    m = np.nanmean(M, axis=0); M[np.isnan(M)] = np.take(m, np.where(np.isnan(M))[1])
    M -= M.mean(0); M /= (M.std(0, ddof=0) + 1e-8)
    return M

Z = stdize(df[Z_cols].values)
W = stdize(df[W_cols].values)

X = X.values.astype(np.float32)
A = A.astype(np.float32); Y = Y.astype(np.float32)

n, dx = X.shape
dz, dw = Z.shape[1], W.shape[1]
print(f"Loaded RHC: n={n}, dimX={dx}, dimZ={dz}, dimW={dw}")



# 使用 numpy.float64 提高数值稳定性
X_np = X.astype(np.float64)
Z_np = Z.astype(np.float64)
W_np = W.astype(np.float64)
A_np = A.astype(np.float64)
Y_np = Y.astype(np.float64)

n = X_np.shape[0]
ones = np.ones((n, 1), dtype=np.float64)

# 线性基
b_zx = np.concatenate([ones, Z_np, X_np], axis=1)  # n x d1
b_wx = np.concatenate([ones, W_np, X_np], axis=1)  # n x d2
d1, d2 = b_zx.shape[1], b_wx.shape[1]

# 分块特征（phi, psi）
phi_lin = np.concatenate([(1.0 - A_np) * b_zx, A_np * b_zx], axis=1)   # n x (2*d1)
psi_lin = np.concatenate([(1.0 - A_np) * b_wx, A_np * b_wx], axis=1)   # n x (2*d2)

# Tpsi（与 A 无关）：二元 ATE 下相当于 “(a=1) − (a=0)”
Tpsi_lin = np.concatenate([-b_wx, +b_wx], axis=1)                      # n x (2*d2)

# 样本矩
M_hat = (phi_lin.T @ psi_lin) / n                     # (2*d1) x (2*d2)
v_hat = (phi_lin * Y_np).mean(axis=0)                 # (2*d1,)
c_hat = Tpsi_lin.mean(axis=0)                         # (2*d2,)

# 闭式求解
M_pinv   = np.linalg.pinv(M_hat, rcond=1e-8)          # (2*d2) x (2*d1)
theta_h  = M_pinv @ v_hat                             # (2*d2,) —— h 的线性系数
g_per    = Tpsi_lin @ theta_h                         # (n,)    —— 逐样本贡献 (Th)(W,X)

LC_point = float(g_per.mean())
psi_lc   = g_per - LC_point
LC_se    = float(np.std(psi_lc, ddof=1) / np.sqrt(n))
LC_ci    = (LC_point - 1.96 * LC_se, LC_point + 1.96 * LC_se)

print("---------------------------------------------------------------")
print(f"Linear-closed : {LC_point:+.4f} ({LC_se:.5f})")
print(f"95% CIs       : [{LC_ci[0]:+.4f}, {LC_ci[1]:+.4f}]")
print("===============================================================")









############################第一种setting

import numpy as np
import pandas as pd
from linearmodels.iv import IV2SLS

formula = "Y ~ 1 + A + X + [W ~ Z]"

iv_model = IV2SLS.from_formula(formula, data=df)
iv_res = iv_model.fit(cov_type="robust")
############################第一种setting
# print("original:")

# A 是 exog 中第 2 列（intercept, A, X...）
beta_A = iv_res.params[1]
se_A   = iv_res.std_errors[1]

# print("A coefficient:", beta_A)
# print("Std Error    :", se_A)
print(f"2sls(original): {beta_A:.4f} ({se_A:.5f})")
print(f"{beta_A:.4f} ({se_A:.5f}) [{beta_A - 1.96*se_A:.4f}, {beta_A + 1.96*se_A:.4f}]")


############################第二种setting


# ================== Step 1: select top-3 X by correlation with Y ==================

# Y 是 (n, 1)，拉平
Y_flat = Y.reshape(-1)

# 计算每个 X_j 与 Y 的皮尔逊相关（绝对值）
corrs = np.array([
    np.corrcoef(X[:, j], Y_flat)[0, 1] for j in range(X.shape[1])
])

# 处理极端情况（常数列等）
corrs = np.nan_to_num(corrs, nan=0.0)

# 选绝对值最大的 3 个
top_k = 2
top_idx = np.argsort(np.abs(corrs))[-top_k:][::-1]

# print("Top-3 X indices (by |corr(X,Y)|):", top_idx)
# print("Top-3 correlations:", corrs[top_idx])

# 取子矩阵
X_top = X[:, top_idx]    # shape (n, 3)


# ================== Step 2: construct interaction terms ==================

# W: (n, 2), X_top: (n, 3)
WX_top = np.hstack([
    W[:, j:j+1] * X_top for j in range(W.shape[1])
])   # shape (n, 2*3 = 6)

# Z: (n, 2)
ZX_top = np.hstack([
    Z[:, j:j+1] * X_top for j in range(Z.shape[1])
])   # shape (n, 2*3 = 6)

# ================== Step 3: IV 2SLS with reduced interactions ==================

from linearmodels.iv import IV2SLS

# exogenous regressors: [1, A, X]
exog = np.hstack([
    np.ones((len(Y), 1)),
    A.reshape(-1, 1),
    X
])

# endogenous regressors: [W, W×X_top]
endog = np.hstack([
    W,
    WX_top
])

# instruments: [Z, Z×X_top]
instruments = np.hstack([
    Z,
    ZX_top
])

iv_res = IV2SLS(
    dependent=Y,
    exog=exog,
    endog=endog,
    instruments=instruments
).fit(cov_type="robust")

# A 是 exog 的第 2 列（[1, A, X...]）
beta_A = iv_res.params[1]
se_A   = iv_res.std_errors[1]

print("==============================================")
# print("2SLS with reduced interactions (top-3 X)")
print(f"2sls(interact-top3): {beta_A:.4f} ({se_A:.5f})")
print(f"{beta_A:.4f} ({se_A:.5f}) [{beta_A - 1.96*se_A:.4f}, {beta_A + 1.96*se_A:.4f}]")

print("==============================================")












# ######################第三种setting
# import numpy as np
# from linearmodels.iv import IV2SLS
# import statsmodels.api as sm

# 手动构造平方项
X_top2 = X_top ** 2
W2 = W ** 2
Z2 = Z ** 2

exog = np.hstack([
    np.ones((len(Y), 1)),
    A.reshape(-1, 1),
    X,
   # X_top2
])

endog = np.hstack([
    W, 
    W2
    
])

instruments = np.hstack([
    Z,
    Z2
    
])

iv_res = IV2SLS(
    dependent=Y,
    exog=exog,
    endog=endog,
    instruments=instruments
).fit(cov_type="robust")

beta_A = iv_res.params[1]
se_A   = iv_res.std_errors[1]
# print(f"2sls(interact-top3): {beta_A:.4f} ({se_A:.5f})")
print(f"2sls(quar): {beta_A:.4f} ({se_A:.5f})")
print(f"{beta_A:.4f} ({se_A:.5f}) [{beta_A - 1.96*se_A:.4f}, {beta_A + 1.96*se_A:.4f}]")


############################第四种setting
# Interaction + Quadratic IV
# Y ~ A + X + [W, W^2, W*X ~ Z, Z^2, Z*X]

# 交互项
WX = np.hstack([
    W[:, j:j+1] * X for j in range(W.shape[1])
])

ZX = np.hstack([
    Z[:, j:j+1] * X for j in range(Z.shape[1])
])

# 二次项
W2 = W ** 2
Z2 = Z ** 2

# exogenous regressors
exog = np.hstack([
    np.ones((len(Y), 1)),   # intercept
    A.reshape(-1, 1),
    X
])

# endogenous regressors
endog = np.hstack([
    W,
    W2,
    WX_top
])

# instruments
instruments = np.hstack([
    Z,
    Z2,
    ZX_top
])

iv_res = IV2SLS(
    dependent=Y,
    exog=exog,
    endog=endog,
    instruments=instruments
).fit(cov_type="robust")

beta_A = iv_res.params[1]
se_A   = iv_res.std_errors[1]
print("==============================================")
print(f"2sls(interact-quar-both): {beta_A:.4f} ({se_A:.5f})")
print(f"{beta_A:.4f} ({se_A:.5f}) [{beta_A - 1.96*se_A:.4f}, {beta_A + 1.96*se_A:.4f}]")






当前使用的方案 3:
Z_cols = ['pafi1', 'hema1']
W_cols = ['paco21', 'ph1']
Loaded RHC: n=5735, dimX=70, dimZ=2, dimW=2
---------------------------------------------------------------
Linear-closed : -0.0615 (0.00514)
95% CIs       : [-0.0715, -0.0514]
2sls(original): -0.0465 (0.02084)
-0.0465 (0.02084) [-0.0873, -0.0056]


/var/folders/6m/31x0jbln169_6g4fn60b2dsc0000gn/T/ipykernel_18637/1988952886.py:147: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  beta_A = iv_res.params[1]
/var/folders/6m/31x0jbln169_6g4fn60b2dsc0000gn/T/ipykernel_18637/1988952886.py:148: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  se_A   = iv_res.std_errors[1]
/var/folders/6m/31x0jbln169_6g4fn60b2dsc0000gn/T/ipykernel_18637/1988952886.py:226: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.ilo

2sls(interact-top3): -0.0292 (0.02712)
-0.0292 (0.02712) [-0.0823, 0.0240]


/var/folders/6m/31x0jbln169_6g4fn60b2dsc0000gn/T/ipykernel_18637/1988952886.py:283: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  beta_A = iv_res.params[1]
/var/folders/6m/31x0jbln169_6g4fn60b2dsc0000gn/T/ipykernel_18637/1988952886.py:284: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  se_A   = iv_res.std_errors[1]


2sls(quar): 0.0590 (0.14345)
0.0590 (0.14345) [-0.2222, 0.3401]
2sls(interact-quar-both): 0.3148 (0.75826)
0.3148 (0.75826) [-1.1714, 1.8010]


/var/folders/6m/31x0jbln169_6g4fn60b2dsc0000gn/T/ipykernel_18637/1988952886.py:335: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  beta_A = iv_res.params[1]
/var/folders/6m/31x0jbln169_6g4fn60b2dsc0000gn/T/ipykernel_18637/1988952886.py:336: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  se_A   = iv_res.std_errors[1]


In [4]:
# ---------------- 6.5) Linear-Closed baseline (closed-form bridge; ATE, plug-in SE) ----------------
# 公式： J_hat = E_n[Tpsi]^T · (E_n[phi psi^T])^+ · E_n[Y phi]
# plug-in 方差：以 REG 形式的逐样本 g_i = (Tpsi_i)^T theta_h 构造 sd(g_i - mean)/sqrt(n)

import numpy as np


import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F, torch.optim as optim
from math import sqrt

# ---------------- 1) Load & preprocess ----------------
url = "https://hbiostat.org/data/repo/rhc.csv"   # Vanderbilt Biostatistics RHC
df = pd.read_csv(url)

# A = RHC 指示；Y = 30-day survival (和论文表方向一致)
A = (df["swang1"].astype(str).str.upper().str.strip() == "RHC").astype(np.float32).values[:, None]
Y = (df["dth30"].astype(str).str.upper().str.strip() == "NO").astype(np.float32).values[:, None]

# 负控变量（与论文设置一致）
Z_cols = ["pafi1", "paco21"]   # NC exposures
W_cols = ["ph1",   "hema1"]    # NC outcomes





from itertools import combinations
# 所有变量
all_cols = ["pafi1", "paco21", "ph1", "hema1"]
# 生成所有 6 种分配方案：从 4 个里选 2 个做 Z，其余做 W
schemes = []
for z_pair in combinations(all_cols, 2):
    w_pair = [col for col in all_cols if col not in z_pair]
    schemes.append((list(z_pair), w_pair))
# === 这里用一个参数来控制采用哪一种方案（1~6）===
scheme_id =  4  # 改成 1,2,3,4,5,6 之一
if not (1 <= scheme_id <= len(schemes)):
    raise ValueError(f"scheme_id 必须在 1~{len(schemes)} 之间")
Z_cols, W_cols = schemes[scheme_id - 1]
print(f"当前使用的方案 {scheme_id}:")
print("Z_cols =", Z_cols)
print("W_cols =", W_cols)





for c in Z_cols + W_cols:
    if c not in df.columns:
        raise ValueError(f"Missing column {c} in RHC csv")

# X = baseline 协变量（除去 A、Y、W、Z 及明显非特征列）
drop_cols = ["ptid","sadmdte","dschdte","dthdte","lstctdte","t3d30","swang1","dth30"] + Z_cols + W_cols
X_df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors="ignore")

# One-hot（drop_first 以避免共线），均值填补 + 标准化
num_cols = X_df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X_df.columns if c not in num_cols]
X = pd.get_dummies(X_df, columns=cat_cols, drop_first=True)
X = X.replace([np.inf, -np.inf], np.nan).fillna(X.mean()).astype(np.float32)
X = ((X - X.mean())/X.std(ddof=0)).fillna(0.0)

# W, Z 简单均值填补 + 标准化
def stdize(M):
    M = M.astype(np.float32).copy()
    m = np.nanmean(M, axis=0); M[np.isnan(M)] = np.take(m, np.where(np.isnan(M))[1])
    M -= M.mean(0); M /= (M.std(0, ddof=0) + 1e-8)
    return M

Z = stdize(df[Z_cols].values)
W = stdize(df[W_cols].values)

X = X.values.astype(np.float32)
A = A.astype(np.float32); Y = Y.astype(np.float32)

n, dx = X.shape
dz, dw = Z.shape[1], W.shape[1]
print(f"Loaded RHC: n={n}, dimX={dx}, dimZ={dz}, dimW={dw}")



# 使用 numpy.float64 提高数值稳定性
X_np = X.astype(np.float64)
Z_np = Z.astype(np.float64)
W_np = W.astype(np.float64)
A_np = A.astype(np.float64)
Y_np = Y.astype(np.float64)

n = X_np.shape[0]
ones = np.ones((n, 1), dtype=np.float64)

# 线性基
b_zx = np.concatenate([ones, Z_np, X_np], axis=1)  # n x d1
b_wx = np.concatenate([ones, W_np, X_np], axis=1)  # n x d2
d1, d2 = b_zx.shape[1], b_wx.shape[1]

# 分块特征（phi, psi）
phi_lin = np.concatenate([(1.0 - A_np) * b_zx, A_np * b_zx], axis=1)   # n x (2*d1)
psi_lin = np.concatenate([(1.0 - A_np) * b_wx, A_np * b_wx], axis=1)   # n x (2*d2)

# Tpsi（与 A 无关）：二元 ATE 下相当于 “(a=1) − (a=0)”
Tpsi_lin = np.concatenate([-b_wx, +b_wx], axis=1)                      # n x (2*d2)

# 样本矩
M_hat = (phi_lin.T @ psi_lin) / n                     # (2*d1) x (2*d2)
v_hat = (phi_lin * Y_np).mean(axis=0)                 # (2*d1,)
c_hat = Tpsi_lin.mean(axis=0)                         # (2*d2,)

# 闭式求解
M_pinv   = np.linalg.pinv(M_hat, rcond=1e-8)          # (2*d2) x (2*d1)
theta_h  = M_pinv @ v_hat                             # (2*d2,) —— h 的线性系数
g_per    = Tpsi_lin @ theta_h                         # (n,)    —— 逐样本贡献 (Th)(W,X)

LC_point = float(g_per.mean())
psi_lc   = g_per - LC_point
LC_se    = float(np.std(psi_lc, ddof=1) / np.sqrt(n))
LC_ci    = (LC_point - 1.96 * LC_se, LC_point + 1.96 * LC_se)

print("---------------------------------------------------------------")
print(f"Linear-closed : {LC_point:+.4f} ({LC_se:.5f})")
print(f"95% CIs       : [{LC_ci[0]:+.4f}, {LC_ci[1]:+.4f}]")
print("===============================================================")









############################第一种setting

import numpy as np
import pandas as pd
from linearmodels.iv import IV2SLS

formula = "Y ~ 1 + A + X + [W ~ Z]"

iv_model = IV2SLS.from_formula(formula, data=df)
iv_res = iv_model.fit(cov_type="robust")
############################第一种setting
# print("original:")

# A 是 exog 中第 2 列（intercept, A, X...）
beta_A = iv_res.params[1]
se_A   = iv_res.std_errors[1]

# print("A coefficient:", beta_A)
# print("Std Error    :", se_A)
print(f"2sls(original): {beta_A:.4f} ({se_A:.5f})")
print(f"{beta_A:.4f} ({se_A:.5f}) [{beta_A - 1.96*se_A:.4f}, {beta_A + 1.96*se_A:.4f}]")


############################第二种setting


# ================== Step 1: select top-3 X by correlation with Y ==================

# Y 是 (n, 1)，拉平
Y_flat = Y.reshape(-1)

# 计算每个 X_j 与 Y 的皮尔逊相关（绝对值）
corrs = np.array([
    np.corrcoef(X[:, j], Y_flat)[0, 1] for j in range(X.shape[1])
])

# 处理极端情况（常数列等）
corrs = np.nan_to_num(corrs, nan=0.0)

# 选绝对值最大的 3 个
top_k = 2
top_idx = np.argsort(np.abs(corrs))[-top_k:][::-1]

# print("Top-3 X indices (by |corr(X,Y)|):", top_idx)
# print("Top-3 correlations:", corrs[top_idx])

# 取子矩阵
X_top = X[:, top_idx]    # shape (n, 3)


# ================== Step 2: construct interaction terms ==================

# W: (n, 2), X_top: (n, 3)
WX_top = np.hstack([
    W[:, j:j+1] * X_top for j in range(W.shape[1])
])   # shape (n, 2*3 = 6)

# Z: (n, 2)
ZX_top = np.hstack([
    Z[:, j:j+1] * X_top for j in range(Z.shape[1])
])   # shape (n, 2*3 = 6)

# ================== Step 3: IV 2SLS with reduced interactions ==================

from linearmodels.iv import IV2SLS

# exogenous regressors: [1, A, X]
exog = np.hstack([
    np.ones((len(Y), 1)),
    A.reshape(-1, 1),
    X
])

# endogenous regressors: [W, W×X_top]
endog = np.hstack([
    W,
    WX_top
])

# instruments: [Z, Z×X_top]
instruments = np.hstack([
    Z,
    ZX_top
])

iv_res = IV2SLS(
    dependent=Y,
    exog=exog,
    endog=endog,
    instruments=instruments
).fit(cov_type="robust")

# A 是 exog 的第 2 列（[1, A, X...]）
beta_A = iv_res.params[1]
se_A   = iv_res.std_errors[1]

print("==============================================")
# print("2SLS with reduced interactions (top-3 X)")
print(f"2sls(interact-top3): {beta_A:.4f} ({se_A:.5f})")
print(f"{beta_A:.4f} ({se_A:.5f}) [{beta_A - 1.96*se_A:.4f}, {beta_A + 1.96*se_A:.4f}]")

print("==============================================")












# ######################第三种setting
# import numpy as np
# from linearmodels.iv import IV2SLS
# import statsmodels.api as sm

# 手动构造平方项
X_top2 = X_top ** 2
W2 = W ** 2
Z2 = Z ** 2

exog = np.hstack([
    np.ones((len(Y), 1)),
    A.reshape(-1, 1),
    X,
   # X_top2
])

endog = np.hstack([
    W, 
    W2
    
])

instruments = np.hstack([
    Z,
    Z2
    
])

iv_res = IV2SLS(
    dependent=Y,
    exog=exog,
    endog=endog,
    instruments=instruments
).fit(cov_type="robust")

beta_A = iv_res.params[1]
se_A   = iv_res.std_errors[1]
# print(f"2sls(interact-top3): {beta_A:.4f} ({se_A:.5f})")
print(f"2sls(quar): {beta_A:.4f} ({se_A:.5f})")
print(f"{beta_A:.4f} ({se_A:.5f}) [{beta_A - 1.96*se_A:.4f}, {beta_A + 1.96*se_A:.4f}]")


############################第四种setting
# Interaction + Quadratic IV
# Y ~ A + X + [W, W^2, W*X ~ Z, Z^2, Z*X]

# 交互项
WX = np.hstack([
    W[:, j:j+1] * X for j in range(W.shape[1])
])

ZX = np.hstack([
    Z[:, j:j+1] * X for j in range(Z.shape[1])
])

# 二次项
W2 = W ** 2
Z2 = Z ** 2

# exogenous regressors
exog = np.hstack([
    np.ones((len(Y), 1)),   # intercept
    A.reshape(-1, 1),
    X
])

# endogenous regressors
endog = np.hstack([
    W,
    W2,
    WX_top
])

# instruments
instruments = np.hstack([
    Z,
    Z2,
    ZX_top
])

iv_res = IV2SLS(
    dependent=Y,
    exog=exog,
    endog=endog,
    instruments=instruments
).fit(cov_type="robust")

beta_A = iv_res.params[1]
se_A   = iv_res.std_errors[1]
print("==============================================")
print(f"2sls(interact-quar-both): {beta_A:.4f} ({se_A:.5f})")
print(f"{beta_A:.4f} ({se_A:.5f}) [{beta_A - 1.96*se_A:.4f}, {beta_A + 1.96*se_A:.4f}]")






当前使用的方案 4:
Z_cols = ['paco21', 'ph1']
W_cols = ['pafi1', 'hema1']
Loaded RHC: n=5735, dimX=70, dimZ=2, dimW=2
---------------------------------------------------------------
Linear-closed : +0.9134 (0.05901)
95% CIs       : [+0.7978, +1.0291]
2sls(original): -0.0827 (0.05068)
-0.0827 (0.05068) [-0.1821, 0.0166]


/var/folders/6m/31x0jbln169_6g4fn60b2dsc0000gn/T/ipykernel_18637/2113422612.py:147: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  beta_A = iv_res.params[1]
/var/folders/6m/31x0jbln169_6g4fn60b2dsc0000gn/T/ipykernel_18637/2113422612.py:148: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  se_A   = iv_res.std_errors[1]
/var/folders/6m/31x0jbln169_6g4fn60b2dsc0000gn/T/ipykernel_18637/2113422612.py:226: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.ilo

2sls(interact-top3): -0.0437 (0.02342)
-0.0437 (0.02342) [-0.0896, 0.0022]
2sls(quar): -0.0701 (0.03614)
-0.0701 (0.03614) [-0.1409, 0.0008]
2sls(interact-quar-both): -0.0875 (0.04659)
-0.0875 (0.04659) [-0.1788, 0.0039]


/var/folders/6m/31x0jbln169_6g4fn60b2dsc0000gn/T/ipykernel_18637/2113422612.py:283: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  beta_A = iv_res.params[1]
/var/folders/6m/31x0jbln169_6g4fn60b2dsc0000gn/T/ipykernel_18637/2113422612.py:284: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  se_A   = iv_res.std_errors[1]
/var/folders/6m/31x0jbln169_6g4fn60b2dsc0000gn/T/ipykernel_18637/2113422612.py:335: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.ilo

In [5]:
# ---------------- 6.5) Linear-Closed baseline (closed-form bridge; ATE, plug-in SE) ----------------
# 公式： J_hat = E_n[Tpsi]^T · (E_n[phi psi^T])^+ · E_n[Y phi]
# plug-in 方差：以 REG 形式的逐样本 g_i = (Tpsi_i)^T theta_h 构造 sd(g_i - mean)/sqrt(n)

import numpy as np


import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F, torch.optim as optim
from math import sqrt

# ---------------- 1) Load & preprocess ----------------
url = "https://hbiostat.org/data/repo/rhc.csv"   # Vanderbilt Biostatistics RHC
df = pd.read_csv(url)

# A = RHC 指示；Y = 30-day survival (和论文表方向一致)
A = (df["swang1"].astype(str).str.upper().str.strip() == "RHC").astype(np.float32).values[:, None]
Y = (df["dth30"].astype(str).str.upper().str.strip() == "NO").astype(np.float32).values[:, None]

# 负控变量（与论文设置一致）
Z_cols = ["pafi1", "paco21"]   # NC exposures
W_cols = ["ph1",   "hema1"]    # NC outcomes





from itertools import combinations
# 所有变量
all_cols = ["pafi1", "paco21", "ph1", "hema1"]
# 生成所有 6 种分配方案：从 4 个里选 2 个做 Z，其余做 W
schemes = []
for z_pair in combinations(all_cols, 2):
    w_pair = [col for col in all_cols if col not in z_pair]
    schemes.append((list(z_pair), w_pair))
# === 这里用一个参数来控制采用哪一种方案（1~6）===
scheme_id =  5  # 改成 1,2,3,4,5,6 之一
if not (1 <= scheme_id <= len(schemes)):
    raise ValueError(f"scheme_id 必须在 1~{len(schemes)} 之间")
Z_cols, W_cols = schemes[scheme_id - 1]
print(f"当前使用的方案 {scheme_id}:")
print("Z_cols =", Z_cols)
print("W_cols =", W_cols)





for c in Z_cols + W_cols:
    if c not in df.columns:
        raise ValueError(f"Missing column {c} in RHC csv")

# X = baseline 协变量（除去 A、Y、W、Z 及明显非特征列）
drop_cols = ["ptid","sadmdte","dschdte","dthdte","lstctdte","t3d30","swang1","dth30"] + Z_cols + W_cols
X_df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors="ignore")

# One-hot（drop_first 以避免共线），均值填补 + 标准化
num_cols = X_df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X_df.columns if c not in num_cols]
X = pd.get_dummies(X_df, columns=cat_cols, drop_first=True)
X = X.replace([np.inf, -np.inf], np.nan).fillna(X.mean()).astype(np.float32)
X = ((X - X.mean())/X.std(ddof=0)).fillna(0.0)

# W, Z 简单均值填补 + 标准化
def stdize(M):
    M = M.astype(np.float32).copy()
    m = np.nanmean(M, axis=0); M[np.isnan(M)] = np.take(m, np.where(np.isnan(M))[1])
    M -= M.mean(0); M /= (M.std(0, ddof=0) + 1e-8)
    return M

Z = stdize(df[Z_cols].values)
W = stdize(df[W_cols].values)

X = X.values.astype(np.float32)
A = A.astype(np.float32); Y = Y.astype(np.float32)

n, dx = X.shape
dz, dw = Z.shape[1], W.shape[1]
print(f"Loaded RHC: n={n}, dimX={dx}, dimZ={dz}, dimW={dw}")



# 使用 numpy.float64 提高数值稳定性
X_np = X.astype(np.float64)
Z_np = Z.astype(np.float64)
W_np = W.astype(np.float64)
A_np = A.astype(np.float64)
Y_np = Y.astype(np.float64)

n = X_np.shape[0]
ones = np.ones((n, 1), dtype=np.float64)

# 线性基
b_zx = np.concatenate([ones, Z_np, X_np], axis=1)  # n x d1
b_wx = np.concatenate([ones, W_np, X_np], axis=1)  # n x d2
d1, d2 = b_zx.shape[1], b_wx.shape[1]

# 分块特征（phi, psi）
phi_lin = np.concatenate([(1.0 - A_np) * b_zx, A_np * b_zx], axis=1)   # n x (2*d1)
psi_lin = np.concatenate([(1.0 - A_np) * b_wx, A_np * b_wx], axis=1)   # n x (2*d2)

# Tpsi（与 A 无关）：二元 ATE 下相当于 “(a=1) − (a=0)”
Tpsi_lin = np.concatenate([-b_wx, +b_wx], axis=1)                      # n x (2*d2)

# 样本矩
M_hat = (phi_lin.T @ psi_lin) / n                     # (2*d1) x (2*d2)
v_hat = (phi_lin * Y_np).mean(axis=0)                 # (2*d1,)
c_hat = Tpsi_lin.mean(axis=0)                         # (2*d2,)

# 闭式求解
M_pinv   = np.linalg.pinv(M_hat, rcond=1e-8)          # (2*d2) x (2*d1)
theta_h  = M_pinv @ v_hat                             # (2*d2,) —— h 的线性系数
g_per    = Tpsi_lin @ theta_h                         # (n,)    —— 逐样本贡献 (Th)(W,X)

LC_point = float(g_per.mean())
psi_lc   = g_per - LC_point
LC_se    = float(np.std(psi_lc, ddof=1) / np.sqrt(n))
LC_ci    = (LC_point - 1.96 * LC_se, LC_point + 1.96 * LC_se)

print("---------------------------------------------------------------")
print(f"Linear-closed : {LC_point:+.4f} ({LC_se:.5f})")
print(f"95% CIs       : [{LC_ci[0]:+.4f}, {LC_ci[1]:+.4f}]")
print("===============================================================")









############################第一种setting

import numpy as np
import pandas as pd
from linearmodels.iv import IV2SLS

formula = "Y ~ 1 + A + X + [W ~ Z]"

iv_model = IV2SLS.from_formula(formula, data=df)
iv_res = iv_model.fit(cov_type="robust")
############################第一种setting
# print("original:")

# A 是 exog 中第 2 列（intercept, A, X...）
beta_A = iv_res.params[1]
se_A   = iv_res.std_errors[1]

# print("A coefficient:", beta_A)
# print("Std Error    :", se_A)
print(f"2sls(original): {beta_A:.4f} ({se_A:.5f})")
print(f"{beta_A:.4f} ({se_A:.5f}) [{beta_A - 1.96*se_A:.4f}, {beta_A + 1.96*se_A:.4f}]")


############################第二种setting


# ================== Step 1: select top-3 X by correlation with Y ==================

# Y 是 (n, 1)，拉平
Y_flat = Y.reshape(-1)

# 计算每个 X_j 与 Y 的皮尔逊相关（绝对值）
corrs = np.array([
    np.corrcoef(X[:, j], Y_flat)[0, 1] for j in range(X.shape[1])
])

# 处理极端情况（常数列等）
corrs = np.nan_to_num(corrs, nan=0.0)

# 选绝对值最大的 3 个
top_k = 2
top_idx = np.argsort(np.abs(corrs))[-top_k:][::-1]

# print("Top-3 X indices (by |corr(X,Y)|):", top_idx)
# print("Top-3 correlations:", corrs[top_idx])

# 取子矩阵
X_top = X[:, top_idx]    # shape (n, 3)


# ================== Step 2: construct interaction terms ==================

# W: (n, 2), X_top: (n, 3)
WX_top = np.hstack([
    W[:, j:j+1] * X_top for j in range(W.shape[1])
])   # shape (n, 2*3 = 6)

# Z: (n, 2)
ZX_top = np.hstack([
    Z[:, j:j+1] * X_top for j in range(Z.shape[1])
])   # shape (n, 2*3 = 6)

# ================== Step 3: IV 2SLS with reduced interactions ==================

from linearmodels.iv import IV2SLS

# exogenous regressors: [1, A, X]
exog = np.hstack([
    np.ones((len(Y), 1)),
    A.reshape(-1, 1),
    X
])

# endogenous regressors: [W, W×X_top]
endog = np.hstack([
    W,
    WX_top
])

# instruments: [Z, Z×X_top]
instruments = np.hstack([
    Z,
    ZX_top
])

iv_res = IV2SLS(
    dependent=Y,
    exog=exog,
    endog=endog,
    instruments=instruments
).fit(cov_type="robust")

# A 是 exog 的第 2 列（[1, A, X...]）
beta_A = iv_res.params[1]
se_A   = iv_res.std_errors[1]

print("==============================================")
# print("2SLS with reduced interactions (top-3 X)")
print(f"2sls(interact-top3): {beta_A:.4f} ({se_A:.5f})")
print(f"{beta_A:.4f} ({se_A:.5f}) [{beta_A - 1.96*se_A:.4f}, {beta_A + 1.96*se_A:.4f}]")

print("==============================================")












# ######################第三种setting
# import numpy as np
# from linearmodels.iv import IV2SLS
# import statsmodels.api as sm

# 手动构造平方项
X_top2 = X_top ** 2
W2 = W ** 2
Z2 = Z ** 2

exog = np.hstack([
    np.ones((len(Y), 1)),
    A.reshape(-1, 1),
    X,
   # X_top2
])

endog = np.hstack([
    W, 
    W2
    
])

instruments = np.hstack([
    Z,
    Z2
    
])

iv_res = IV2SLS(
    dependent=Y,
    exog=exog,
    endog=endog,
    instruments=instruments
).fit(cov_type="robust")

beta_A = iv_res.params[1]
se_A   = iv_res.std_errors[1]
# print(f"2sls(interact-top3): {beta_A:.4f} ({se_A:.5f})")
print(f"2sls(quar): {beta_A:.4f} ({se_A:.5f})")
print(f"{beta_A:.4f} ({se_A:.5f}) [{beta_A - 1.96*se_A:.4f}, {beta_A + 1.96*se_A:.4f}]")


############################第四种setting
# Interaction + Quadratic IV
# Y ~ A + X + [W, W^2, W*X ~ Z, Z^2, Z*X]

# 交互项
WX = np.hstack([
    W[:, j:j+1] * X for j in range(W.shape[1])
])

ZX = np.hstack([
    Z[:, j:j+1] * X for j in range(Z.shape[1])
])

# 二次项
W2 = W ** 2
Z2 = Z ** 2

# exogenous regressors
exog = np.hstack([
    np.ones((len(Y), 1)),   # intercept
    A.reshape(-1, 1),
    X
])

# endogenous regressors
endog = np.hstack([
    W,
    W2,
    WX_top
])

# instruments
instruments = np.hstack([
    Z,
    Z2,
    ZX_top
])

iv_res = IV2SLS(
    dependent=Y,
    exog=exog,
    endog=endog,
    instruments=instruments
).fit(cov_type="robust")

beta_A = iv_res.params[1]
se_A   = iv_res.std_errors[1]
print("==============================================")
print(f"2sls(interact-quar-both): {beta_A:.4f} ({se_A:.5f})")
print(f"{beta_A:.4f} ({se_A:.5f}) [{beta_A - 1.96*se_A:.4f}, {beta_A + 1.96*se_A:.4f}]")






当前使用的方案 5:
Z_cols = ['paco21', 'hema1']
W_cols = ['pafi1', 'ph1']
Loaded RHC: n=5735, dimX=70, dimZ=2, dimW=2
---------------------------------------------------------------
Linear-closed : +0.2531 (0.02020)
95% CIs       : [+0.2135, +0.2927]
2sls(original): 0.1560 (0.15349)
0.1560 (0.15349) [-0.1448, 0.4568]


/var/folders/6m/31x0jbln169_6g4fn60b2dsc0000gn/T/ipykernel_18637/2831779266.py:147: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  beta_A = iv_res.params[1]
/var/folders/6m/31x0jbln169_6g4fn60b2dsc0000gn/T/ipykernel_18637/2831779266.py:148: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  se_A   = iv_res.std_errors[1]
/var/folders/6m/31x0jbln169_6g4fn60b2dsc0000gn/T/ipykernel_18637/2831779266.py:226: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.ilo

2sls(interact-top3): -0.1227 (0.21800)
-0.1227 (0.21800) [-0.5500, 0.3046]
2sls(quar): 0.1674 (0.10504)
0.1674 (0.10504) [-0.0384, 0.3733]
2sls(interact-quar-both): 0.0730 (0.37460)
0.0730 (0.37460) [-0.6612, 0.8072]


/var/folders/6m/31x0jbln169_6g4fn60b2dsc0000gn/T/ipykernel_18637/2831779266.py:283: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  beta_A = iv_res.params[1]
/var/folders/6m/31x0jbln169_6g4fn60b2dsc0000gn/T/ipykernel_18637/2831779266.py:284: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  se_A   = iv_res.std_errors[1]
/var/folders/6m/31x0jbln169_6g4fn60b2dsc0000gn/T/ipykernel_18637/2831779266.py:335: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.ilo

In [6]:
# ---------------- 6.5) Linear-Closed baseline (closed-form bridge; ATE, plug-in SE) ----------------
# 公式： J_hat = E_n[Tpsi]^T · (E_n[phi psi^T])^+ · E_n[Y phi]
# plug-in 方差：以 REG 形式的逐样本 g_i = (Tpsi_i)^T theta_h 构造 sd(g_i - mean)/sqrt(n)

import numpy as np


import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F, torch.optim as optim
from math import sqrt

# ---------------- 1) Load & preprocess ----------------
url = "https://hbiostat.org/data/repo/rhc.csv"   # Vanderbilt Biostatistics RHC
df = pd.read_csv(url)

# A = RHC 指示；Y = 30-day survival (和论文表方向一致)
A = (df["swang1"].astype(str).str.upper().str.strip() == "RHC").astype(np.float32).values[:, None]
Y = (df["dth30"].astype(str).str.upper().str.strip() == "NO").astype(np.float32).values[:, None]

# 负控变量（与论文设置一致）
Z_cols = ["pafi1", "paco21"]   # NC exposures
W_cols = ["ph1",   "hema1"]    # NC outcomes





from itertools import combinations
# 所有变量
all_cols = ["pafi1", "paco21", "ph1", "hema1"]
# 生成所有 6 种分配方案：从 4 个里选 2 个做 Z，其余做 W
schemes = []
for z_pair in combinations(all_cols, 2):
    w_pair = [col for col in all_cols if col not in z_pair]
    schemes.append((list(z_pair), w_pair))
# === 这里用一个参数来控制采用哪一种方案（1~6）===
scheme_id =  6  # 改成 1,2,3,4,5,6 之一
if not (1 <= scheme_id <= len(schemes)):
    raise ValueError(f"scheme_id 必须在 1~{len(schemes)} 之间")
Z_cols, W_cols = schemes[scheme_id - 1]
print(f"当前使用的方案 {scheme_id}:")
print("Z_cols =", Z_cols)
print("W_cols =", W_cols)





for c in Z_cols + W_cols:
    if c not in df.columns:
        raise ValueError(f"Missing column {c} in RHC csv")

# X = baseline 协变量（除去 A、Y、W、Z 及明显非特征列）
drop_cols = ["ptid","sadmdte","dschdte","dthdte","lstctdte","t3d30","swang1","dth30"] + Z_cols + W_cols
X_df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors="ignore")

# One-hot（drop_first 以避免共线），均值填补 + 标准化
num_cols = X_df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X_df.columns if c not in num_cols]
X = pd.get_dummies(X_df, columns=cat_cols, drop_first=True)
X = X.replace([np.inf, -np.inf], np.nan).fillna(X.mean()).astype(np.float32)
X = ((X - X.mean())/X.std(ddof=0)).fillna(0.0)

# W, Z 简单均值填补 + 标准化
def stdize(M):
    M = M.astype(np.float32).copy()
    m = np.nanmean(M, axis=0); M[np.isnan(M)] = np.take(m, np.where(np.isnan(M))[1])
    M -= M.mean(0); M /= (M.std(0, ddof=0) + 1e-8)
    return M

Z = stdize(df[Z_cols].values)
W = stdize(df[W_cols].values)

X = X.values.astype(np.float32)
A = A.astype(np.float32); Y = Y.astype(np.float32)

n, dx = X.shape
dz, dw = Z.shape[1], W.shape[1]
print(f"Loaded RHC: n={n}, dimX={dx}, dimZ={dz}, dimW={dw}")



# 使用 numpy.float64 提高数值稳定性
X_np = X.astype(np.float64)
Z_np = Z.astype(np.float64)
W_np = W.astype(np.float64)
A_np = A.astype(np.float64)
Y_np = Y.astype(np.float64)

n = X_np.shape[0]
ones = np.ones((n, 1), dtype=np.float64)

# 线性基
b_zx = np.concatenate([ones, Z_np, X_np], axis=1)  # n x d1
b_wx = np.concatenate([ones, W_np, X_np], axis=1)  # n x d2
d1, d2 = b_zx.shape[1], b_wx.shape[1]

# 分块特征（phi, psi）
phi_lin = np.concatenate([(1.0 - A_np) * b_zx, A_np * b_zx], axis=1)   # n x (2*d1)
psi_lin = np.concatenate([(1.0 - A_np) * b_wx, A_np * b_wx], axis=1)   # n x (2*d2)

# Tpsi（与 A 无关）：二元 ATE 下相当于 “(a=1) − (a=0)”
Tpsi_lin = np.concatenate([-b_wx, +b_wx], axis=1)                      # n x (2*d2)

# 样本矩
M_hat = (phi_lin.T @ psi_lin) / n                     # (2*d1) x (2*d2)
v_hat = (phi_lin * Y_np).mean(axis=0)                 # (2*d1,)
c_hat = Tpsi_lin.mean(axis=0)                         # (2*d2,)

# 闭式求解
M_pinv   = np.linalg.pinv(M_hat, rcond=1e-8)          # (2*d2) x (2*d1)
theta_h  = M_pinv @ v_hat                             # (2*d2,) —— h 的线性系数
g_per    = Tpsi_lin @ theta_h                         # (n,)    —— 逐样本贡献 (Th)(W,X)

LC_point = float(g_per.mean())
psi_lc   = g_per - LC_point
LC_se    = float(np.std(psi_lc, ddof=1) / np.sqrt(n))
LC_ci    = (LC_point - 1.96 * LC_se, LC_point + 1.96 * LC_se)

print("---------------------------------------------------------------")
print(f"Linear-closed : {LC_point:+.4f} ({LC_se:.5f})")
print(f"95% CIs       : [{LC_ci[0]:+.4f}, {LC_ci[1]:+.4f}]")
print("===============================================================")









############################第一种setting

import numpy as np
import pandas as pd
from linearmodels.iv import IV2SLS

formula = "Y ~ 1 + A + X + [W ~ Z]"

iv_model = IV2SLS.from_formula(formula, data=df)
iv_res = iv_model.fit(cov_type="robust")
############################第一种setting
# print("original:")

# A 是 exog 中第 2 列（intercept, A, X...）
beta_A = iv_res.params[1]
se_A   = iv_res.std_errors[1]

# print("A coefficient:", beta_A)
# print("Std Error    :", se_A)
print(f"2sls(original): {beta_A:.4f} ({se_A:.5f})")
print(f"{beta_A:.4f} ({se_A:.5f}) [{beta_A - 1.96*se_A:.4f}, {beta_A + 1.96*se_A:.4f}]")


############################第二种setting


# ================== Step 1: select top-3 X by correlation with Y ==================

# Y 是 (n, 1)，拉平
Y_flat = Y.reshape(-1)

# 计算每个 X_j 与 Y 的皮尔逊相关（绝对值）
corrs = np.array([
    np.corrcoef(X[:, j], Y_flat)[0, 1] for j in range(X.shape[1])
])

# 处理极端情况（常数列等）
corrs = np.nan_to_num(corrs, nan=0.0)

# 选绝对值最大的 3 个
top_k = 2
top_idx = np.argsort(np.abs(corrs))[-top_k:][::-1]

# print("Top-3 X indices (by |corr(X,Y)|):", top_idx)
# print("Top-3 correlations:", corrs[top_idx])

# 取子矩阵
X_top = X[:, top_idx]    # shape (n, 3)


# ================== Step 2: construct interaction terms ==================

# W: (n, 2), X_top: (n, 3)
WX_top = np.hstack([
    W[:, j:j+1] * X_top for j in range(W.shape[1])
])   # shape (n, 2*3 = 6)

# Z: (n, 2)
ZX_top = np.hstack([
    Z[:, j:j+1] * X_top for j in range(Z.shape[1])
])   # shape (n, 2*3 = 6)

# ================== Step 3: IV 2SLS with reduced interactions ==================

from linearmodels.iv import IV2SLS

# exogenous regressors: [1, A, X]
exog = np.hstack([
    np.ones((len(Y), 1)),
    A.reshape(-1, 1),
    X
])

# endogenous regressors: [W, W×X_top]
endog = np.hstack([
    W,
    WX_top
])

# instruments: [Z, Z×X_top]
instruments = np.hstack([
    Z,
    ZX_top
])

iv_res = IV2SLS(
    dependent=Y,
    exog=exog,
    endog=endog,
    instruments=instruments
).fit(cov_type="robust")

# A 是 exog 的第 2 列（[1, A, X...]）
beta_A = iv_res.params[1]
se_A   = iv_res.std_errors[1]

print("==============================================")
# print("2SLS with reduced interactions (top-3 X)")
print(f"2sls(interact-top3): {beta_A:.4f} ({se_A:.5f})")
print(f"{beta_A:.4f} ({se_A:.5f}) [{beta_A - 1.96*se_A:.4f}, {beta_A + 1.96*se_A:.4f}]")

print("==============================================")












# ######################第三种setting
# import numpy as np
# from linearmodels.iv import IV2SLS
# import statsmodels.api as sm

# 手动构造平方项
X_top2 = X_top ** 2
W2 = W ** 2
Z2 = Z ** 2

exog = np.hstack([
    np.ones((len(Y), 1)),
    A.reshape(-1, 1),
    X,
   # X_top2
])

endog = np.hstack([
    W, 
    W2
    
])

instruments = np.hstack([
    Z,
    Z2
    
])

iv_res = IV2SLS(
    dependent=Y,
    exog=exog,
    endog=endog,
    instruments=instruments
).fit(cov_type="robust")

beta_A = iv_res.params[1]
se_A   = iv_res.std_errors[1]
# print(f"2sls(interact-top3): {beta_A:.4f} ({se_A:.5f})")
print(f"2sls(quar): {beta_A:.4f} ({se_A:.5f})")
print(f"{beta_A:.4f} ({se_A:.5f}) [{beta_A - 1.96*se_A:.4f}, {beta_A + 1.96*se_A:.4f}]")


############################第四种setting
# Interaction + Quadratic IV
# Y ~ A + X + [W, W^2, W*X ~ Z, Z^2, Z*X]

# 交互项
WX = np.hstack([
    W[:, j:j+1] * X for j in range(W.shape[1])
])

ZX = np.hstack([
    Z[:, j:j+1] * X for j in range(Z.shape[1])
])

# 二次项
W2 = W ** 2
Z2 = Z ** 2

# exogenous regressors
exog = np.hstack([
    np.ones((len(Y), 1)),   # intercept
    A.reshape(-1, 1),
    X
])

# endogenous regressors
endog = np.hstack([
    W,
    W2,
    WX_top
])

# instruments
instruments = np.hstack([
    Z,
    Z2,
    ZX_top
])

iv_res = IV2SLS(
    dependent=Y,
    exog=exog,
    endog=endog,
    instruments=instruments
).fit(cov_type="robust")

beta_A = iv_res.params[1]
se_A   = iv_res.std_errors[1]
print("==============================================")
print(f"2sls(interact-quar-both): {beta_A:.4f} ({se_A:.5f})")
print(f"{beta_A:.4f} ({se_A:.5f}) [{beta_A - 1.96*se_A:.4f}, {beta_A + 1.96*se_A:.4f}]")






当前使用的方案 6:
Z_cols = ['ph1', 'hema1']
W_cols = ['pafi1', 'paco21']
Loaded RHC: n=5735, dimX=70, dimZ=2, dimW=2
---------------------------------------------------------------
Linear-closed : +0.1130 (0.00647)
95% CIs       : [+0.1004, +0.1257]
2sls(original): 0.1031 (0.09324)
0.1031 (0.09324) [-0.0797, 0.2858]


/var/folders/6m/31x0jbln169_6g4fn60b2dsc0000gn/T/ipykernel_18637/175204921.py:147: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  beta_A = iv_res.params[1]
/var/folders/6m/31x0jbln169_6g4fn60b2dsc0000gn/T/ipykernel_18637/175204921.py:148: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  se_A   = iv_res.std_errors[1]
/var/folders/6m/31x0jbln169_6g4fn60b2dsc0000gn/T/ipykernel_18637/175204921.py:226: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[p

2sls(interact-top3): -0.0132 (0.04776)
-0.0132 (0.04776) [-0.1068, 0.0804]
2sls(quar): 0.1576 (0.09945)
0.1576 (0.09945) [-0.0373, 0.3525]
2sls(interact-quar-both): 0.0336 (0.10160)
0.0336 (0.10160) [-0.1656, 0.2327]


/var/folders/6m/31x0jbln169_6g4fn60b2dsc0000gn/T/ipykernel_18637/175204921.py:283: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  beta_A = iv_res.params[1]
/var/folders/6m/31x0jbln169_6g4fn60b2dsc0000gn/T/ipykernel_18637/175204921.py:284: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  se_A   = iv_res.std_errors[1]
/var/folders/6m/31x0jbln169_6g4fn60b2dsc0000gn/T/ipykernel_18637/175204921.py:335: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[p